**/!\ DISCLAIMER** The code of this notebook have been created with help of LLM, Sonnet 4.5

It's been used:
 - to help understand reuse existing code from course materials online such as deeplearning.ai and projectPro to our dataset.
 - to search for state of the art ways to improve the models performance giving the highly imabalanced nature of the dataset.
 - to help with debugging and understanding the models.


# BERT-style Transformer for South Park Character Classification

This notebook implements a custom BERT-style transformer encoder for predicting which South Park character spoke a given line of dialogue.

## Why Transformers?

Transformers have revolutionized NLP by addressing key limitations of RNNs:

- **Parallel Processing** - Unlike RNNs, transformers process entire sequences simultaneously
- **Long-range Dependencies** - Self-attention captures relationships between any two tokens
- **No Vanishing Gradients** - Direct connections between all positions
- **Bidirectional Context** - Understands context from both directions simultaneously

### BERT Architecture:

Our implementation follows BERT's encoder-only design:
1. **Token + Positional Embeddings** - Encode words and their positions
2. **Multi-Head Self-Attention** - Learn different aspects of relationships
3. **Feed-Forward Networks** - Process attended representations
4. **Layer Normalization + Residual Connections** - Stable training
5. **[CLS] Token Classification** - Use first token for sequence classification

For character classification, we expect transformers to capture:
- **Character-specific phrases** - Cartman: Respect my authority
- **Speaking patterns** - Stan: Dude, this is...
- **Contextual nuances** - Word order and emphasis matter

## 1. Imports and Setup

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive',  force_remount=True)

# import os
# os.chdir('/content/drive/MyDrive/Project_SP')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn

import numpy as np
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import sys 
sys.path.append('../utils')



# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Configuration Parameters

In [43]:
# Transformer hyperparameters
EMBEDDING_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 4
HIDDEN_DIM = 512
MAX_SEQ_LEN = 256
DROPOUT_RATE = 0.2

# Training hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 1e-4
MAX_LR = 5e-4
NUM_EPOCHS = 50
WEIGHT_DECAY = 0.01
LABEL_SMOOTHING = 0.1

# Early stopping
EARLY_STOP_PATIENCE = 7

# Text preprocessing
MIN_WORD_FREQ = 2
MAX_VOCAB_SIZE = 10000



print(f"Configuration:")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Num attention heads: {NUM_HEADS}")
print(f"  Num transformer layers: {NUM_LAYERS}")
print(f"  Feedforward dim: {HIDDEN_DIM}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"  Dropout rate: {DROPOUT_RATE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE} -> {MAX_LR}")
print(f"  Max epochs: {NUM_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOP_PATIENCE}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")

Configuration:
  Embedding dim: 128
  Num attention heads: 4
  Num transformer layers: 4
  Feedforward dim: 512
  Max sequence length: 256
  Dropout rate: 0.2
  Batch size: 64
  Learning rate: 0.0001 -> 0.0005
  Max epochs: 50
  Early stopping patience: 7
  Label smoothing: 0.1


## 3. Load Data

In [ ]:
import pandas as pd
# read data from parquet 
df = pd.read_parquet('../01 - Inputs/south_park_dialogues.parquet')

In [ ]:
train_data = pd.read_parquet('../01 - Inputs/train_data_labels_as_numbers.parquet')
val_data = pd.read_parquet('../01 - Inputs/val_data_labels_as_numbers.parquet')
test_data = pd.read_parquet('../01 - Inputs/test_data_labels_as_numbers.parquet')

## 6. Text Preprocessing and Vocabulary

For BERT-style transformers, we need special tokens:
- `<pad>` - Padding token (index 0)
- `<unk>` - Unknown words (index 1)
- `<cls>` - Classification token (index 2) - prepended to every sequence
- `<sep>` - Separator token (index 3) - optional for single sentences

In [47]:
train_texts = train_data['Line'].tolist()
train_labels = train_data['Character'].tolist()

val_texts = val_data['Line'].tolist()
val_labels = val_data['Character'].tolist()

test_texts = test_data['Line'].tolist()
test_labels = test_data['Character'].tolist()

In [50]:
sys.path.insert(0, '/content/drive/MyDrive/Project_SP/utils')
sys.path.insert(0, '/content/drive/MyDrive/Project_SP/03 - Model Building')


In [51]:
from model_utils import preprocess_text, Vocabulary
import sys

# Preprocess all texts
print("Preprocessing texts...")
train_tokens = [preprocess_text(text) for text in tqdm(train_texts, desc="Train")]
val_tokens = [preprocess_text(text) for text in tqdm(val_texts, desc="Val")]
test_tokens = [preprocess_text(text) for text in tqdm(test_texts, desc="Test")]

# Build vocabulary
vocab = Vocabulary(min_freq=MIN_WORD_FREQ)
vocab.build_vocab(train_tokens)

print(f"\nVocabulary size: {len(vocab):,}")
print(f"Special tokens: <pad>={vocab.word2idx['<pad>']}, <unk>={vocab.word2idx['<unk>']}, <cls>={vocab.word2idx['<cls>']}, <sep>={vocab.word2idx['<sep>']}")
print(f"\nMost common words:")
for word, count in vocab.word_counts.most_common(10):
    print(f"  {word}: {count}")

Preprocessing texts...


Test: 100%|██████████| 6036/6036 [00:00<00:00, 548006.82it/s]


Vocabulary size: 8,081
Special tokens: <pad>=0, <unk>=1, <cls>=2, <sep>=3

Most common words:
  you: 10603
  the: 9195
  i: 8060
  to: 7904
  a: 5939
  and: 5259
  it: 4271
  is: 3768
  we: 3744
  that: 3740


## 7. Dataset and DataLoader

For BERT-style models, we prepend the `<cls>` token to each sequence and create attention masks.

In [52]:
from model_utils import create_dataloaders

train_loader, val_loader, test_loader = create_dataloaders(train_tokens, train_labels, val_tokens, val_labels,
                       test_tokens, test_labels, vocab, 'bert')

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")

# Test a batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  input_ids: {sample_batch['input_ids'].shape}")
print(f"  attention_mask: {sample_batch['attention_mask'].shape}")
print(f"  labels: {sample_batch['labels'].shape}")
print(f"\nFirst sequence (with [CLS]):")
print(f"  Tokens: {vocab.decode(sample_batch['input_ids'][0].tolist()[:20])}")

Train batches: 441  Val batches: 95  Test batches: 95

Sample batch shapes:
  input_ids: torch.Size([64, 50])
  attention_mask: torch.Size([64, 50])
  labels: torch.Size([64])

First sequence (with [CLS]):
  Tokens: ['<cls>', 'were', 'in', 'the', 'alien', 'ship', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']


## 8. BERT-style Transformer Model Architecture\n
\n
We implement a custom BERT-style transformer encoder using PyTorch's built-in components:\n
\n
1. **Token Embeddings** - Convert word indices to dense vectors\n
2. **Positional Embeddings** - Add position information (learned, not sinusoidal)\n
3. **Transformer Encoder** - Multi-head self-attention + feedforward layers\n
4. **Classification Head** - Use [CLS] token representation for prediction\n
\n
Key differences from RNN/LSTM:\n
- **Parallel processing** - All tokens processed simultaneously\n
- **Self-attention** - Each token attends to all other tokens\n
- **Positional encoding** - Explicit position information (RNNs get this implicitly)\n
- **No recurrence** - No hidden state passed between time steps

In [ ]:
class TransformerClassifier(nn.Module):
    """BERT-style transformer encoder for sequence classification."""
    
    def __init__(self, vocab_size, embedding_dim=128, num_heads=4, 
                 num_layers=4, hidden_dim=512, num_classes=12, 
                 max_seq_len=256, dropout=0.2):
        super(TransformerClassifier, self).__init__()
        
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len
        
        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Positional embeddings (learned, not sinusoidal like original Transformer)
        self.pos_embedding = nn.Embedding(max_seq_len, embedding_dim)
        
        # Dropout for embeddings
        self.embedding_dropout = nn.Dropout(dropout)
        
        # Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            activation='gelu',  # GELU activation like BERT
            batch_first=True,
            norm_first=False  # Post-norm like original Transformer
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head (like BERT)
        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights following BERT initialization."""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.bias is not None:
                    module.bias.data.zero_()
            elif isinstance(module, nn.Embedding):
                module.weight.data.normal_(mean=0.0, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()
    
    def forward(self, input_ids, attention_mask=None):
        """
        Forward pass.
        
        Args:
            input_ids: (batch_size, seq_len) - Token indices
            attention_mask: (batch_size, seq_len) - 1 for real tokens, 0 for padding
        
        Returns:
            logits: (batch_size, num_classes) - Classification logits
        """
        batch_size, seq_len = input_ids.size()
        
        # Create position indices
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0).expand(batch_size, -1)
        
        # Token + positional embeddings
        token_embeds = self.token_embedding(input_ids)
        pos_embeds = self.pos_embedding(positions)
        embeddings = token_embeds + pos_embeds
        embeddings = self.embedding_dropout(embeddings)
        
        # Create padding mask for transformer (True for padding, False for real tokens)
        # PyTorch transformer expects True for positions to IGNORE
        if attention_mask is not None:
            padding_mask = (attention_mask == 0)
        else:
            padding_mask = None
        
        # Transformer encoding
        encoded = self.transformer(embeddings, src_key_padding_mask=padding_mask)
        
        # Use [CLS] token (first token) for classification
        cls_output = encoded[:, 0, :]  # (batch_size, embedding_dim)
        
        # Classification
        logits = self.classifier(cls_output)
        
        return logits


## model overview
model = TransformerClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(set(train_labels)),
    max_seq_len=MAX_SEQ_LEN,
    dropout=DROPOUT_RATE
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

Model initialized
Total parameters: 1,942,175
Trainable parameters: 1,942,175

Model architecture:
TransformerClassifier(
  (token_embedding): Embedding(8081, 128, padding_idx=0)
  (pos_embedding): Embedding(256, 128)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (classifier): Sequ

## 9. Class Weights

In [79]:
from train_utils import compute_balanced_class_weights

class_weights = compute_balanced_class_weights(train_labels, len(set(train_labels)), device)

print("Class weights computed")
print(f"Class weights: {class_weights}")
print(f"Weight range: {class_weights.min():.3f} - {class_weights.max():.3f}")

Class weights computed
Class weights: tensor([0.1364, 0.1860, 0.1983, 0.5028, 0.5031, 1.2196, 1.4968, 1.5143, 1.7273,
        1.9168, 2.0236, 2.0372, 2.2490, 2.3119, 2.3661, 2.3723, 3.1880, 3.4945,
        3.7390, 4.2064, 4.2260, 4.3061, 4.4979, 4.6356, 4.6594, 4.6834, 5.0198,
        5.4082, 5.9384, 6.1391, 6.2232], device='cuda:0')
Weight range: 0.136 - 6.223


## 10. Training Function with Early Stopping

Training enhancements for transformers:
- **AdamW optimizer** - Adam with decoupled weight decay (better for transformers)
- **OneCycleLR scheduler** - Learning rate warmup + cosine annealing
- **Gradient clipping** - Prevents exploding gradients
- **Label smoothing** - Reduces overconfidence
- **Early stopping** - Stops when validation loss plateaus

**cliping gradient** important for transformers as it helps to prevent the exploding gradient problem.
https://apxml.com/courses/foundations-transformers-architecture/chapter-7-implementation-details-optimization/gradient-clipping-transformers


In [ ]:
def train_transformer(model, train_loader, val_loader, num_epochs, lr, max_lr, 
                     weight_decay, class_weights, label_smoothing, criterion, optimizer, scheduler,patience=7):
    """
    Train transformer model with advanced optimization techniques.
    
    Args:
        model: Transformer model
        train_loader: Training data loader
        val_loader: Validation data loader
        num_epochs: Maximum number of epochs
        lr: Initial learning rate
        max_lr: Maximum learning rate for OneCycleLR
        weight_decay: L2 regularization strength
        class_weights: Class weights for imbalanced data
        label_smoothing: Label smoothing factor
        patience: Early stopping patience
    
    Returns:
        Dictionary with model, history, and metrics
    """
    model.to(device)
    
    
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_f1': [], 'val_acc': [], 'lr': []}
    best_val_loss = float('inf')
    best_f1 = 0.0
    best_model_state = None
    patience_counter = 0
    
    print("Training BERT-style Transformer...")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_preds, train_labels_list = [], []
        
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for batch in train_pbar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            scheduler.step()  # Update learning rate every batch
            
            # Track metrics
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())
            
            # Update progress bar
            current_lr = scheduler.get_last_lr()[0]
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{current_lr:.2e}'})
        
        train_loss /= len(train_loader)
        train_acc = accuracy_score(train_labels_list, train_preds)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_preds, val_labels_list = [], []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids, attention_mask)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels_list, val_preds)
        val_f1 = f1_score(val_labels_list, val_preds, average='macro')
        
        # Record history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(scheduler.get_last_lr()[0])
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Train Acc={train_acc:.4f} | "
              f"Val Loss={val_loss:.4f} Val Acc={val_acc:.4f} | Val F1={val_f1:.4f} | LR={scheduler.get_last_lr()[0]:.2e}")
        
        # Early stopping
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"  -> New best model (val_f1: {val_f1:.4f})")
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    model.load_state_dict(best_model_state)
    elapsed = time.time() - start_time
    print(f"Training completed in {elapsed/60:.2f} minutes")
    
    return {'model': model, 'history': history, 'best_val_loss': best_val_loss, 'time': elapsed}

## 11. Train Model

In [117]:
from train_utils import create_training_components
loss_types = ['label_smoothing_ce', 'cross_entropy', 'focal']

components = {}
models = {}
for loss_type in loss_types:
    models[loss_type] = TransformerClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    hidden_dim=HIDDEN_DIM,
    num_classes=len(set(train_labels)),
    max_seq_len=MAX_SEQ_LEN,
    dropout=DROPOUT_RATE
).to(device)
    components[loss_type] = create_training_components(
        model=models[loss_type],
        train_labels=train_labels,  
        K=len(set(train_labels)),
        device=device,
        optimizer_type='adamw',
        lr=LEARNING_RATE,
        loss_type=loss_type,
        scheduler_type='onecycle',
        weight_decay=WEIGHT_DECAY,
        max_lr=MAX_LR,
        epochs=NUM_EPOCHS,
        steps_per_epoch=len(train_loader)
    )


In [118]:
# Train the transformer model
results = {}

for loss_type, component in components.items():
    print(f"Training with {loss_type} loss...")
    
    results[loss_type] = train_transformer(
        models[loss_type],
        train_loader,
        val_loader,
        NUM_EPOCHS,
        LEARNING_RATE,
        MAX_LR,
        WEIGHT_DECAY,
        class_weights,
        LABEL_SMOOTHING,
        component['criterion'],
        component['optimizer'],
        component['scheduler'],
        EARLY_STOP_PATIENCE
    )


Training with label_smoothing_ce loss...
Training BERT-style Transformer...


Epoch 1/50 [Val]: 100%|██████████| 95/95 [00:00<00:00, 526.65it/s]


Epoch 1: Train Loss=4.1485 Train Acc=0.0076 | Val Loss=4.1376 Val Acc=0.0053 | Val F1=0.0003402300805920003 | LR=6.59e-05
  -> New best model (val_f1: 0.0003)


Epoch 2/50 [Val]: 100%|██████████| 95/95 [00:00<00:00, 525.62it/s]


Epoch 2: Train Loss=4.1125 Train Acc=0.0174 | Val Loss=4.0682 Val Acc=0.0189 | Val F1=0.00838185251353946 | LR=1.86e-04
  -> New best model (val_f1: 0.0084)


...

Epoch 23: Train Loss=0.0238 Train Acc=0.4892 | Val Loss=0.1744 Val Acc=0.2174 | Val F1=0.17146957650675076 | LR=3.27e-04


Epoch 24/50 [Val]: 100%|██████████| 95/95 [00:00<00:00, 523.73it/s]


Epoch 24: Train Loss=0.0231 Train Acc=0.4939 | Val Loss=0.1769 Val Acc=0.2184 | Val F1=0.17110714332925 | LR=3.10e-04


Epoch 25/50 [Val]: 100%|██████████| 95/95 [00:00<00:00, 520.14it/s]


Epoch 25: Train Loss=0.0224 Train Acc=0.5019 | Val Loss=0.1683 Val Acc=0.2094 | Val F1=0.17268688591177056 | LR=2.93e-04
Early stopping at epoch 25
Training completed in 1.10 minutes


## 12. Evaluate Model
Evaluate the trained transformer on the test set with comprehensive metrics.

In [131]:
import sys
sys.path.append('../utils')
from evaluate_models import plot_confusion_matrix, print_evaluation_report, evaluate_model

In [ ]:
from evaluate_models import COLOR_DICT, SP_BLUE, SP_ORANGE, SP_GREEN, SP_BROWN, SP_RED

def evaluate_transformer(model, loader):
    """Evaluate transformer model on a dataset."""
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_score': f1_score(all_labels, all_preds, average='macro', zero_division=0)
    }
    
    return metrics, all_preds, all_labels


test_metrics_df = []
def plot_training_history(result, result_type):
    # Plot training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    axes[0].plot(result['history']['train_loss'], label='Train Loss', marker='o', color=SP_GREEN, linewidth=2)
    axes[0].plot(result['history']['val_loss'], label='Val Loss', marker='s', color=SP_RED, linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title(f'Training and Validation Loss - {result_type} Loss', fontweight='bold', fontsize=14)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3, color=SP_BROWN, linestyle='--')

    # Accuracy curves
    axes[1].plot(result['history']['train_acc'], label='Train Accuracy', marker='o', color=SP_GREEN, linewidth=2)
    axes[1].plot(result['history']['val_acc'], label='Val Accuracy', marker='s', color=SP_RED, linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy', fontsize=12)
    axes[1].set_title(f'Training and Validation Accuracy - {result_type} Loss', fontweight='bold', fontsize=14)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3, color=SP_BROWN, linestyle='--')

    plt.tight_layout()
    plt.savefig(f'transformer_bert_k{len(set(train_labels))}_{result_type}_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Plot learning rate schedule
    plt.figure(figsize=(10, 4))
    plt.plot(result['history']['lr'], color=SP_GREEN, linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Learning Rate', fontsize=12)
    plt.title(f'Learning Rate Schedule (OneCycleLR with Warmup) - {result_type} Loss', fontweight='bold', fontsize=14)
    plt.grid(True, alpha=0.3, color=SP_BROWN, linestyle='--')
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig(f'transformer_bert_k{len(set(train_labels))}_{result_type}_lr_schedule.png', dpi=150, bbox_inches='tight')
    plt.show()
    


label_to_char = pickle.load(open('../01 - Inputs/label_to_char.pkl', 'rb'))

y_true = [label_to_char[i] for i in test_labels_list]

best_recall = 0.0   
for loss_type, result in results.items():
    model = result['model']
    test_metrics, test_preds, test_labels_list = evaluate_transformer(model, test_loader)
    if best_recall < test_metrics['recall']:
        best_recall = test_metrics['recall']
        best_model = model
        best_loss_type = loss_type
        best_test_metrics = test_metrics
        best_test_preds = test_preds
        best_test_labels_list = test_labels_list
    # Print results
    print("=" * 80)
    print("BERT-STYLE TRANSFORMER - TEST SET PERFORMANCE")
    print("=" * 80)
    print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
    print(f"Precision: {test_metrics['precision']:.4f}")
    print(f"Recall:    {test_metrics['recall']:.4f}")
    print(f"F1-Score:  {test_metrics['f1_score']:.4f}")
    print("=" * 80)
    
    # Plot training history
    plot_training_history(result, loss_type)
    
    y_pred =  [label_to_char[i] for i in test_preds]
    plot_confusion_matrix(y_true, y_pred, save_path=f'transformer_bert_k{len(set(train_labels))}_loss_{loss_type} _confusion_matrix.png', title=f"Confusion Matrix transformer_bert_k{len(set(train_labels))} - {loss_type} Loss")
    
    test_metrics_df.append(pd.DataFrame(test_metrics, index=[f'Transformer BERT {loss_type}']))
print(f"Best model: {best_loss_type} with recall: {best_recall:.4f}")

TypeError: 'TransformerClassifier' object is not subscriptable

In [ ]:
pd.concat(test_metrics_df).to_csv(f'transformer_bert_k{len(set(train_labels))}_{loss_type}_test_metrics.csv')

## 13. Save Model

Save the trained transformer model with full configuration and metrics.

In [ ]:
# Save checkpoint
model = best_model
test_metrics = best_test_metrics
results = results[best_loss_type]
label_to_char = pickle.load(open('01 - Inputs/label_to_char.pkl', 'rb'))
char_to_label = pickle.load(open('01 - Inputs/char_to_label.pkl', 'rb'))

checkpoint = {
    'model_type': 'BERT-style Transformer',
    'model_state_dict': model.state_dict(),
    'config': {
        'K': len,
        'vocab_size': len(vocab),
        'embedding_dim': EMBEDDING_DIM,
        'num_heads': NUM_HEADS,
        'num_layers': NUM_LAYERS,
        'hidden_dim': HIDDEN_DIM,
        'max_seq_len': MAX_SEQ_LEN,
        'dropout_rate': DROPOUT_RATE,
        'label_smoothing': LABEL_SMOOTHING,
        'weight_decay': WEIGHT_DECAY
    },
    'char_to_label': char_to_label,
    'label_to_char': label_to_char,
    'vocab': {
        'word2idx': vocab.word2idx,
        'idx2word': vocab.idx2word
    },
    'test_metrics': test_metrics,
    'history': results['history'],
    'training_time': results['time'],
    'total_parameters': sum(p.numel() for p in model.parameters())
}

filename = f'transformer_bert_k{len(set(train_labels))}.pt'
torch.save(checkpoint, filename)

print(f"Model saved to: {filename}")
print(f"\nModel Summary:")
print(f"  Architecture: BERT-style Transformer Encoder")
print(f"  Parameters: {checkpoint['total_parameters']:,}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Attention heads: {NUM_HEADS}")
print(f"  Transformer layers: {NUM_LAYERS}")
print(f"  Feedforward dim: {HIDDEN_DIM}")
print(f"  Vocabulary size: {len(vocab):,}")
print(f"  Max sequence length: {MAX_SEQ_LEN}")
print(f"\nPerformance:")
print(f"  Test Accuracy: {test_metrics['accuracy']:.4f}")
print(f"  Test F1-Score: {test_metrics['f1_score']:.4f}")
print(f"  Training time: {results['time']/60:.2f} minutes")
print(f"\nComparison with other models:")
print(f"  Load this checkpoint and compare metrics with:")
print(f"    - baseline_logistic_regression.ipynb")
print(f"    - rnn_lstm_gru_models.ipynb")
print(f"    - seq2seq_basic.ipynb")

Model saved to: transformer_bert_k31.pt

Model Summary:
  Architecture: BERT-style Transformer Encoder
  Parameters: 1,942,175
  Embedding dim: 128
  Attention heads: 4
  Transformer layers: 4
  Feedforward dim: 512
  Vocabulary size: 8,081
  Max sequence length: 256

Performance:
  Test Accuracy: 0.2099
  Test F1-Score: 0.1792
  Training time: 0.87 minutes

Comparison with other models:
  Load this checkpoint and compare metrics with:
    - baseline_logistic_regression.ipynb
    - rnn_lstm_gru_models.ipynb
    - seq2seq_basic.ipynb
